# Biblical Gemma 4 E4B Fine-Tuning with Unsloth (4-bit QLoRA, Mobile Profile)

**Base Model:** Gemma 4 E4B Instruct

**Dataset:** Per-persona datagen JSONL — each persona has its own system prompt and distinctive voice

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** Mobile-oriented profile (smaller adapter + shorter context for lower memory)

**Chat Template:** Tokenizer-native Gemma chat template (applied via `tokenizer.apply_chat_template`)

**Architecture:** This base LoRA teaches the model persona-switching — "when the system prompt says you're Amos, speak like Amos; when it says David, speak like David." Optional persona LoRAs can refine individual voices further.

## 1. Configuration

All paths and variables for easy configuration.

In [1]:

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/workspace/training/biblical"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"
MODEL_NAME_BASE = "biblical_gemma4_e4b_unsloth_4bit_mobile"

# =========================== INPUT DATA ===========================
# Combined multi-turn ShareGPT JSONL from datagen notebooks
# Already quality-filtered, multi-turn (4 QA pairs per conversation), grouped by topic
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/biblical_persona_v2/biblical_personas_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see data loading cell below).
# This keeps training in sync with datagen — if you regenerate data with new/changed
# prompts, the training notebook picks them up automatically.
# After loading, the dict `persona_system_prompts` maps persona_key -> full prompt text.
# It is also saved alongside the LoRA adapters for use at inference time.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
TARGET_EPOCHS = 1

# =========================== LoRA CONFIGURATION ===========================
# Adapter is merged into the base weights at GGUF export, so adapter size on
# disk is irrelevant. Use full attention + MLP targets and a higher rank for
# stronger persona-distinction learning.
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0

# Full "all-linear" Unsloth recipe: attention + MLP projections.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("✓ Configuration loaded (Gemma 4 E4B 4-bit QLoRA mobile profile)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_FILE}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA output: {LORA_OUTPUT_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Training precision: 4-bit QLoRA")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Persona prompts: extracted from JSONL at load time")


✓ Configuration loaded (Gemma 4 E4B 4-bit QLoRA mobile profile)
  Base model: unsloth/gemma-4-E4B-it-unsloth-bnb-4bit
  Model name: biblical_gemma4_e4b_unsloth_4bit_mobile
  Input data: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_sharegpt.jsonl
  Output base: /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile
  LoRA output: /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/lora_adapters
  LoRA config: r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Training: batch=2, grad_accum=4, lr=0.0002
  Training precision: 4-bit QLoRA
  Max seq length: 2048
  Persona prompts: extracted from JSONL at load time


## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [2]:
# Install core packages in the running notebook container
!pip install -q -U unsloth trl accelerate datasets bitsandbytes

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel ≥0.16 on PyPI, so
# uninstall — peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
!pip uninstall -y -q torchao

# Gemma 4 support may be ahead of PyPI releases.
!pip install -q -U git+https://github.com/huggingface/transformers.git

# Keep PEFT compatible with latest Transformers main.
!pip install -q -U git+https://github.com/huggingface/peft.git

# Verify installations
import importlib.util
import unsloth
import transformers
import peft
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ PEFT: {peft.__version__}")
print(f"✓ TRL: {trl.__version__}")
print(f"✓ torchao installed: {importlib.util.find_spec('torchao') is not None} (should be False)")
print("Environment ready. Restart kernel, then rerun from Cell 3.")


  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.9 requires torchao>=0.13.0, which is not installed.
unsloth-zoo 2026.4.9 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.7.0.dev0 which is incompatible.
unsloth 2026.4.8 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=

## 3. Load Dataset

Load the combined multi-turn ShareGPT JSONL from `biblical_datagen.ipynb`.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: 4 QA pairs grouped per conversation by topic
- Each conversation has a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [3]:

import json, os, re
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to snake_case key: lowercase, strip leading "the ", underscores for spaces
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            print(f"  ⚠️ Could not extract persona from system prompt: {sys_msg[:80]}...")

dataset = HFDataset.from_list(conversations)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")


LOADING COMBINED SHAREGPT DATA
  File: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_sharegpt.jsonl

Total dataset: 2612 multi-turn conversations across 26 personas
Extracted 26 unique system prompts from JSONL
Columns: ['conversations']

Per-persona breakdown:
  paul                   199 conversations
  isaiah                 199 conversations
  david                  199 conversations
  jeremiah               198 conversations
  moses                  198 conversations
  daniel                 197 conversations
  job                    192 conversations
  ezekiel                192 conversations
  solomon                169 conversations
  peter                  126 conversations
  zechariah              120 conversations
  hosea                   87 conversations
  amos                    76 conversations
  joshua                  72 conversations
  apostle_john            60 conversations
  micah                   60 conversations
  james   

## 4. Validate & Summarize Dataset

Datagen data is already clean (no artifacts to strip). Verify data quality and show persona distribution.

In [4]:

bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count ≥3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\n⚠️ Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\n⚠️ Filtering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 50) + "▌" * (1 if count % 50 >= 25 else 0)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    # Extract persona name from system prompt "You are X, ..."
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\n✓ Dataset validated and ready for training")


DATA QUALITY CHECK
  Total examples: 2612
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 26 (should match extracted count: 26)
  Turn distribution: {5: 91, 7: 541, 9: 1980}

PERSONA DISTRIBUTION:
  paul           199  ███▌
  isaiah         199  ███▌
  david          199  ███▌
  jeremiah       198  ███▌
  moses          198  ███▌
  daniel         197  ███▌
  job            192  ███▌
  ezekiel        192  ███▌
  solomon        169  ███
  peter          126  ██▌
  zechariah      120  ██
  hosea           87  █▌
  amos            76  █▌
  joshua          72  █
  apostle_john    60  █
  micah           60  █
  james           44  ▌
  malachi         36  ▌
  joel            36  ▌
  zephaniah       31  ▌
  habakkuk        28  ▌
  jonah           24  
  nahum           21  
  haggai          20  
  obadiah         16  
  jude            12  
  TOTAL         2612

VOICE SAMPLES (first ~100 chars of response):
  Paul: "When I was still breathing threats and hurled my fury again

## 5. Load Model & Tokenizer (4-bit)

Load Gemma 4 E4B in 4-bit precision for QLoRA training.

- **DGX Spark (128GB):** ample headroom for training and experimentation
- **Mobile profile:** reduced context + smaller adapter for easier downstream deployment

In [5]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and broken
# torch.compile path. Without these, Gemma falls back to slow eager Python loops.
# Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Gemma 4 can return a Processor object instead of a plain tokenizer.
if hasattr(tokenizer, "vocab_size"):
    vocab_size = tokenizer.vocab_size
elif hasattr(tokenizer, "tokenizer") and hasattr(tokenizer.tokenizer, "vocab_size"):
    vocab_size = tokenizer.tokenizer.vocab_size
else:
    vocab_size = "unknown"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit (QLoRA)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {vocab_size}")
print(f"  Attn impl: {getattr(model.config, '_attn_implementation', 'unknown')}")


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.7.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: `flash_attention_2` is not supported for `gemma4` because max attention head dim 512 exceeds the Flash Attention 2 limit of 256 - defaulting to `sdpa`.


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✓ Model loaded: unsloth/gemma-4-E4B-it-unsloth-bnb-4bit
  Precision: 4-bit (QLoRA)
  Max sequence length: 2048
  Vocab size: 262144
  Attn impl: sdpa


## 6. Format Dataset for Chat Template

Apply Gemma's tokenizer chat template to each conversation and build the final training dataset.

In [6]:
# Standardize ShareGPT rows, then format with Gemma tokenizer chat template
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset
dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples and shuffle
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
dataset = dataset.shuffle(seed=42)

print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n\u2713 Dataset formatted: {len(dataset)} examples")

Unsloth: Standardizing formats (num_proc=24):   0%|          | 0/2612 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2612 [00:00<?, ? examples/s]

--- Sample formatted text (first 500 chars) ---
<bos><|turn>system
You are Moses, the liberator and lawgiver who spoke with God face to face, led your people out of Egypt through the sea and wilderness, and received the covenant on Sinai.

YOUR DISTINCTIVE VOICE: Authoritative but reluctant. Legal precision in laws, epic narrative in storytelling. Speaks as one burdened by an entire nation's weight. Uses 'Hear, O Israel' and direct divine quotation ('And the LORD said unto Moses'). Stuttering origins yet thundering delivery. Both tender fathe

✓ Dataset formatted: 2612 examples


## 7. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See Step 1 config for module options.

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"\u2713 LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"\u2713 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


✓ LoRA adapters added (r=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
✓ Trainable parameters: 81,166,336


## 8. Trainer Setup

- 1 epoch, low learning rate (1e-4) — gently teaches persona-switching without degrading base capabilities
- Each example has a persona-specific system prompt so the model learns distinct voices

In [8]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
print(f"✓ Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} × {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: enabled")
print(f"  Dataset: {len(dataset)} examples")


Unsloth: Sample packing skipped (processor-based model detected).


Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/2612 [00:00<?, ? examples/s]

✓ Trainer configured
  Effective batch size: 2 × 4 = 8
  Epochs: 1
  LR: 0.0002
  Packing: enabled
  Dataset: 2612 examples


## 9. Train

In [9]:
# Start training
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,612 | Num Epochs = 1 | Total steps = 327
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 81,166,336 of 8,022,267,168 (1.01% trained)
[transformers] Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.913470
10,0.673912
15,0.593784
20,0.557729
25,0.526606
30,0.501303
35,0.483166
40,0.479759
45,0.469233
50,0.457252


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/train/checkpoint-327/tokenizer_config.json.


TrainOutput(global_step=327, training_loss=0.43868700043506215, metrics={'train_runtime': 7550.1269, 'train_samples_per_second': 0.346, 'train_steps_per_second': 0.043, 'total_flos': 1.373507510215968e+17, 'train_loss': 0.43868700043506215, 'epoch': 1.0})

## 10. Save LoRA Adapters

Save the trained LoRA adapters. These can be loaded on compatible Gemma 4 E4B models with PEFT or served via vLLM.

In [10]:

from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

print(f"\n✓ LoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["amos"]  # or any persona key')


Saving LoRA adapters to /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/lora_adapters...


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/lora_adapters/tokenizer_config.json.



✓ LoRA adapters saved!
  Adapters:       /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/lora_adapters
  System prompts: /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/lora_adapters/persona_system_prompts.json (26 personas)

  At inference, load prompts with:
    with open("/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/lora_adapters/persona_system_prompts.json") as f:
        prompts = json.load(f)
    system_msg = prompts["amos"]  # or any persona key


## 11. Test Inference

Quick smoke test with a few personas using their extracted system prompts. Each persona should respond in its distinctive voice.


In [11]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST — {len(test_personas)} PERSONAS\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.95,
        top_k=64,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs


INFERENCE TEST — 4 PERSONAS



TypeError: 'NoneType' object is not subscriptable

## 12. Verify Adapter (Reload from Disk)

Final validation: load the adapter cold from disk to confirm it's self-contained and portable.


In [12]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("✓ Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

# Reload saved system prompts
import json
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

# Test with first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.95,
    top_k=64,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\n✓ Adapter loads cleanly from disk. Ready for deployment via vLLM.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()


✓ Cleared training model from memory
  Loading adapter from: /workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/lora_adapters
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.7.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

TypeError: 'NoneType' object is not subscriptable

## 13. Export Merged Model to GGUF (Mobile / Cross-Platform)

Merge the LoRA adapter into the base model and export to GGUF so it can
run anywhere llama.cpp / Ollama / LM Studio / iOS GGUF runners are supported
(e.g. iPhone via apps like LLMFarm, PocketPal, Private LLM).

- `q4_k_m` is the recommended quant for phones (best size/quality tradeoff).
- `q5_k_m` and `q8_0` are also produced for higher-quality desktop use.
- Requires Unsloth's bundled `llama.cpp` build (downloaded automatically on
  first call to `save_pretrained_gguf`).

**Note:** Gemma 4 GGUF export requires a recent llama.cpp version that
supports the Gemma 4 architecture. If conversion fails, update llama.cpp
or wait for upstream support to land.

In [ ]:
# Reload adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model2, tokenizer2
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter will merge before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# Pass ALL quant methods in one call so the LoRA→FP16 merge happens ONCE
# and llama.cpp quantizes from that single merged file. Otherwise the loop
# re-merges and re-writes the FP16 GGUF for every quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge → {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\n\u2713 GGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(Path(GGUF_OUTPUT_DIR).glob("*.gguf")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:60s} {size_mb:>8.1f} MB")

print("\nMobile usage (iPhone):")
print("  1. Transfer the q4_k_m .gguf file to your phone (AirDrop / Files app).")
print("  2. Open in an iOS GGUF runner (LLMFarm, PocketPal, Private LLM, etc.).")
print("  3. Use Gemma chat template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.7.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Merge process complete. Saved to `/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf_gguf/gemma-4-e4b-it.BF16.gguf', '/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf_gguf/gemma-4-e4b-it.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf1

Unsloth: Copying 1 files from cache to `/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf`: 100%|██████████| 1/1 [00:04<00:00,  4.46s/it]


Successfully copied all 1 files from cache to `/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:21<00:00, 81.34s/it]


Unsloth: Merge process complete. Saved to `/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf_gguf/gemma-4-e4b-it.BF16.gguf', '/workspace/training/biblical/output/biblical_gemma4_e4b_unsloth_4bit_mobile/gguf_gguf/gemma-4-e4b-it.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 